In [1]:
import scanpy as sc
import pandas as pd
import scipy.io as sio

In [9]:
adata = sc.read_h5ad("raw/ReplogleWeissman2022_K562_essential.h5ad")

In [10]:
adata

AnnData object with n_obs × n_vars = 310385 × 8563
    obs: 'batch', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'guide_id', 'percent_mito', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'disease', 'cancer', 'cell_line', 'sex', 'age', 'perturbation', 'organism', 'perturbation_type', 'tissue_type', 'ncounts', 'ngenes', 'nperts', 'percent_ribo'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells'

In [24]:
adata.obs["perturbation"].value_counts()

perturbation
control    10691
RPL3        1996
NCBP2        992
KIF11        974
SLC39A9      752
           ...  
RFFL           7
POLR3A         6
RBM22          5
POT1           5
SEC62          5
Name: count, Length: 2058, dtype: int64

In [3]:
adata_2 = sc.read_h5ad("raw/SrivatsanTrapnell2020_sciplex3.h5ad")

In [4]:
adata_2

AnnData object with n_obs × n_vars = 799317 × 110983
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type', 'organism', 'perturbation_type'
    var: 'ensembl_id'

In [44]:
adata_2.obs["time"].value_counts()

time
24.0    680685
72.0     82110
Name: count, dtype: int64

In [9]:
adata_2[adata_2.obs["time"]==24.0].obs["perturbation"].value_counts()

perturbation
control                                15494
Baricitinib (LY3009104, INCB028050)     4315
Tranylcypromine (2-PCPA) HCl            4291
WP1066                                  4263
RG108                                   4244
                                       ...  
Alvespimycin (17-DMAG) HCl              2089
Patupilone (EPO906, Epothilone B)       1822
Flavopiridol HCl                        1729
Epothilone A                            1426
YM155 (Sepantronium Bromide)            1007
Name: count, Length: 189, dtype: int64

In [34]:
obs = adata_2.obs.copy()
obs["plate_well"] = obs["plate"].astype(str) + "_" + obs["well"].astype(str)

wells_per_perturbation = (
    obs.groupby("perturbation")["plate_well"]
    .nunique()
    .sort_values(ascending=False)
)

print(wells_per_perturbation)


perturbation
control                         104
(+)-JQ1                          32
AT9283                           32
Alisertib (MLN8237)              32
2-Methoxyestradiol (2-MeOE2)     32
                               ... 
WHI-P154                         24
XAV-939                          24
WP1066                           24
YM155 (Sepantronium Bromide)     24
Zileuton                         24
Name: plate_well, Length: 189, dtype: int64


/tmp/ipykernel_2175063/2401730795.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs.groupby("perturbation")["plate_well"]


In [37]:
control_layout = (
    obs.loc[obs["perturbation"] != "control"]
    .groupby(["cell_line", "replicate", "time"])["plate_well"]
    .nunique()
    .reset_index(name="n_wells")
    .sort_values(["cell_line", "replicate", "time"])
)

print(control_layout)


   cell_line replicate  time  n_wells
0       A549      rep1  24.0      752
1       A549      rep1  72.0      188
2       A549      rep2  24.0      752
3       A549      rep2  72.0      188
4       K562      rep1  24.0      752
5       K562      rep1  72.0        0
6       K562      rep2  24.0      752
7       K562      rep2  72.0        0
8       MCF7      rep1  24.0      752
9       MCF7      rep1  72.0        0
10      MCF7      rep2  24.0      752
11      MCF7      rep2  72.0        0


/tmp/ipykernel_2175063/3346081904.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["cell_line", "replicate", "time"])["plate_well"]


In [38]:
obs.loc[obs["time"] == 72, ["cell_line", "replicate", "perturbation", "plate_well"]].drop_duplicates() \
   .groupby(["cell_line", "replicate", "perturbation"])["plate_well"].nunique()


/tmp/ipykernel_2175063/4183511726.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["cell_line", "replicate", "perturbation"])["plate_well"].nunique()


cell_line  replicate  perturbation                
A549       rep1       2-Methoxyestradiol (2-MeOE2)    4
                      (+)-JQ1                         4
                      A-366                           0
                      ABT-737                         0
                      AC480 (BMS-599626)              0
                                                     ..
MCF7       rep2       XAV-939                         0
                      YM155 (Sepantronium Bromide)    0
                      ZM 447439                       0
                      Zileuton                        0
                      control                         0
Name: plate_well, Length: 1134, dtype: int64

In [27]:
adata_3 = sc.read_h5ad("raw/NormanWeissman2019_filtered.h5ad")

In [28]:
adata_3

AnnData object with n_obs × n_vars = 111445 × 33694
    obs: 'guide_id', 'read_count', 'UMI_count', 'coverage', 'gemgroup', 'good_coverage', 'number_of_cells', 'tissue_type', 'cell_line', 'cancer', 'disease', 'perturbation_type', 'celltype', 'organism', 'perturbation', 'nperts', 'ngenes', 'ncounts', 'percent_mito', 'percent_ribo'
    var: 'ensemble_id', 'ncounts', 'ncells'

In [29]:
adata_3.obs["perturbation"].value_counts()

perturbation
control          11855
KLF1              1960
BAK1              1457
CEBPE             1233
CEBPE_RUNX1T1     1219
                 ...  
CBL_UBASH3A         64
CEBPB_CEBPA         64
C3orf72_FOXL2       59
JUN_CEBPB           59
JUN_CEBPA           54
Name: count, Length: 237, dtype: int64

In [3]:
adata_4 = sc.read_h5ad("raw/PapalexiSatija2021_eccite_arrayed_RNA.h5ad")

In [4]:
adata_4

AnnData object with n_obs × n_vars = 8984 × 16826
    obs: 'perturbation', 'hto', 'guide_id', 'hto_barcode', 'gdo_barcode', 'tissue_type', 'cell_line', 'cancer', 'disease', 'perturbation_type', 'celltype', 'organism', 'nperts', 'ngenes', 'ncounts', 'percent_mito', 'percent_ribo'
    var: 'ensembl_id', 'ncounts', 'ncells'

In [5]:
adata_4.obs["perturbation"].value_counts()

perturbation
control     2009
ETV7        1789
IRF1         994
ATF2         794
IRF7         750
MARCH8       723
IFNGR1       701
STAT2        576
CAV1         409
PDL1         235
IFNGR2         4
CMTM6          0
CD86           0
CUL3           0
BRD4           0
PDCD1LG2       0
POU2F2         0
NFKBIA         0
JAK2           0
SPI1           0
SMAD4          0
STAT3          0
STAT1          0
STAT5A         0
TNFRSF14       0
UBE2L6         0
eGFP           0
Name: count, dtype: int64

In [45]:
adata_5 = sc.read_h5ad("raw/AissaBenevolenskaya2021.h5ad")

In [46]:
adata_5

AnnData object with n_obs × n_vars = 119071 × 17820
    obs: 'GEO', 'time', 'cell_line', 'perturbation', 'batch', 'subseries', 'replicate', 'tissue_type', 'cancer', 'perturbation_type', 'disease', 'celltype', 'organism', 'nperts', 'ncounts', 'ngenes', 'percent_mito', 'percent_ribo', 'chembl-ID'
    var: 'mt', 'ribo', 'ncounts', 'ncells'

In [47]:
adata_5.obs["perturbation"].value_counts()

perturbation
control                   29983
osimertinib               29963
crizotinib                29785
osimertinib+crizotinib    29340
Name: count, dtype: int64

In [58]:
adata_5.obs["time"].value_counts()

time
72    119071
Name: count, dtype: int64

In [66]:
adata_5.obs[['subseries', 'cell_line', 'time', 'perturbation']].drop_duplicates().sort_values(['perturbation'])


,subseries,cell_line,time,perturbation
cell_barcode,,,,
AAAAACCCGGAA-GSM4869650,GSE160244,PC9_xenograft,72,control
AAAAAACCACCT-GSM4869651,GSE160244,PC9_xenograft,72,crizotinib
AAAAAACTGTCT-GSM4869652,GSE160244,PC9_xenograft,72,osimertinib
AAAAAAAAACCC-GSM4869653,GSE160244,PC9_xenograft,72,osimertinib+crizotinib


In [10]:
adata_6 = sc.read_h5ad("raw/ChangYe2021.h5ad")

In [11]:
adata_6

AnnData object with n_obs × n_vars = 42277 × 45066
    obs: 'sample', 'disease', 'cancer', 'sex', 'perturbation', 'dose_value', 'dose_unit', 'perturbation_type', 'organism', 'nperts', 'ncounts', 'ngenes', 'percent_mito', 'percent_ribo', 'chembl-ID'
    var: 'ensembl_gene_id', 'ncounts', 'ncells'

In [12]:
adata_6.obs["perturbation"].value_counts()

perturbation
control      21043
GNE-104       8737
erlotinib     6577
GNE-069       5920
Name: count, dtype: int64

In [24]:
adata_6.obs["perturbation_type"].value_counts()

perturbation_type
drug    42277
Name: count, dtype: int64

In [25]:
adata_7 = sc.read_h5ad("raw/ZhaoSims2021.h5ad")

In [26]:
adata_7

AnnData object with n_obs × n_vars = 165748 × 60725
    obs: 'sample', 'GEO', 'Sample', 'tissue', 'age', 'sex', 'location', 'diagnosis', 'library', 'dose_value', 'dose_unit', 'perturbation', 'tissue_type', 'cancer', 'disease', 'celltype', 'organism', 'perturbation_type', 'ncounts', 'ngenes', 'percent_mito', 'percent_ribo', 'nperts', 'chembl-ID'
    var: 'ncounts', 'ncells'

In [27]:
adata_7.obs["perturbation"].value_counts()

perturbation
control         88313
etoposide       36513
panobinostat    22220
Ana-12           7085
RO4929097        4806
Tazemetostat     4128
Ispenisib        2683
Name: count, dtype: int64

In [53]:
adata_7.obs["sample"].value_counts()

sample
PW030    50909
PW034    33861
PW036    22505
PW032    20751
PW040    16269
PW029     6150
PW053     5436
PW052     4883
PW051     3808
PW031     1176
Name: count, dtype: int64

In [55]:
adata_7.obs.groupby('sample')[['age', 'sex', 'diagnosis', 'location']].agg(lambda x: x.unique().tolist())

/tmp/ipykernel_326098/2470985634.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata_7.obs.groupby('sample')[['age', 'sex', 'diagnosis', 'location']].agg(lambda x: x.unique().tolist())


,age,sex,diagnosis,location
sample,,,,
PW029,[52],[f],"[Glioblastoma, WHO Grade IV]",[splenial extension into left parietal]
PW030,[65],[m],"[Glioblastoma, WHO Grade IV]",[right parietal]
PW031,[61],[m],"[Glioblastoma, WHO Grade IV]",[left frontal]
PW032,[61],[m],"[Glioblastoma, WHO Grade IV]",[left frontal]
PW034,[68],[f],"[Glioblastoma, WHO Grade IV]",[left parieto-occipital]
PW036,[56],[m],"[Glioblastoma, WHO Grade IV]",[right temporal]
PW040,[69],[m],"[Glioblastoma, WHO Grade IV]",[right temporal]
PW051,[74],[f],"[Glioblastoma, WHO Grade IV, recurrent]",[right frontal]
PW052,[74],[f],"[Glioblastoma, WHO Grade IV, recurrent]",[right frontal]
